# 29 · Total within groups, checked against a pandas groupby

## What this checks

**Total within groups** adds up a table's numbers within each group. This check covers release 0.7.0.

The South London two-by-two table is made of six numbers. They are the populations and cholera deaths of 32
sub-districts, totalled within the three water-supply groups of Snow's Table XII. Until 26 October 2026 these six were
worked out by hand and typed into the example. Fieldwork computes them now. That is why they are worth recomputing by
somebody else.

## How it is checked

We total the same table with **pandas**, Python's table library, by other authors. Its `groupby(...).sum()` does the
same job, and it is what a practitioner would otherwise use. No code is shared with Fieldwork.

The table has 64 rows. The `sub_district` column is the one dropped, so 64 rows become 6 groups.

**The case worth the check is one where the two are meant to disagree.** We empty one cell on purpose. Fieldwork then
returns *nothing known* for that group, and names it. pandas, by default, adds up the cells that are left.

## What should happen

1. **All six totals** exactly equal to pandas. They are whole numbers with nothing approximated, so the tolerance is
   zero, not small.
2. **The shape.** 6 groups from 64 rows, with the `sub_district` column reported as dropped.
3. **With one cell emptied,** that group's total is empty. It equals pandas adding up each group without skipping
   empty cells. The other five totals are unchanged.
4. **The gap is named,** with how many rows were added up and how many were empty. Together these must make the
   group's row count, so a practitioner can find the cell.
5. **What the default would hide.** We record, in people, how far pandas' default total is off. We record what one
   tempting option returns beside it. Neither is required to equal Fieldwork: one is what the rule costs, the other
   is what it is mistaken for.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_29_group_table import run

report = run(Path.cwd() / "fixtures" / "group-snow-001.json")
print("agrees:", report["agrees"])

whole = report["comparisons"][0]
print(f"\n{whole['rowsIn']} rows -> {whole['groupsOut']} groups, dropping {whole['dropped']}\n")
for t in whole["totals"]:
    print(f"  {'OK ' if t['agrees'] else 'DIFF'} {t['pandas']:>7}  {' · '.join(t['group'])}")
for name, ok in whole["checks"].items():
    print(f"    {'OK ' if ok else 'DIFF'} {name}")


## The result on record

First recorded on 27 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.7.0:
**Agrees**.

All six totals are exact:

- **Southwark and Vauxhall only:** 167,654 people and 844 deaths.
- **Both companies:** 300,149 people and 652 deaths.
- **Lambeth only:** 19,133 people and 18 deaths.

With St. Saviour's population emptied, that group returns nothing known. That matches pandas adding up each group
without skipping empty cells. pandas' default returns 147,945, short by 19,709 people. The gap is reported as 11 rows
added up and 1 empty, against the group's 12 rows.

Running the cell above repeats the check on the same data.

## The case worth the check: one empty cell

Any table in Fieldwork can hold an empty cell. Adding up over one gives a total that is **too small by an unknown
amount**, with no warning. A group whose population nobody recorded comes out smaller, not unknown. Fieldwork instead
returns *nothing known* for that group, and names it.

**pandas has no option for this rule.** The check found that by reaching for the obvious one and watching it fail.
The option `min_count=1` reads as "at least one value required", and it is the first thing anyone writes here. But it
sets how many values must be **present**. It does not notice values that are **absent**. So on eleven numbers and one
empty cell it returns the eleven: the default's answer, from the option named for missing data.

Four ways to add up the emptied group:

- **`sum()`** gives 147,945. This is the default. It is short by one sub-district, and nothing says so.
- **`sum(min_count=1)`** gives 147,945. This is the trap: it reads like the rule and behaves like the default.
- **`sum(min_count=12)`** gives nothing known. That is right here, but only because this group has twelve rows. It
  would also empty every smaller group.
- **`sum(skipna=False)`, group by group,** gives nothing known. **This is the comparison.** It is empty exactly where
  a cell was empty, and the group size plays no part.

A check written against the default would have reported a disagreement and called it a defect. So would one written
against `min_count=1`, for a more embarrassing reason. We record this because the next author will reach for
that option too.

In [ ]:
holed = report["comparisons"][1]
e, hidden = holed["emptied"], holed["whatTheDefaultWouldHide"]

print("emptied cell :", e["key"], "was", e["wasValue"])
print("group        :", " · ".join(holed["group"]))
print()
print(f"  Fieldwork                   {holed['fieldwork']}")
print(f"  pandas sum(skipna=False)    {holed['pandasSkipnaFalse']}   <- the comparison")
print(f"  pandas sum()                {holed['pandasDefault']}")
print(f"  pandas sum(min_count=1)     {holed['pandasMinCount1']}   <- not the rule")
print(f"  pandas sum(min_count=12)    {holed['pandasMinCountGroupSize']}   <- right by accident")
print()
print(holed["whyNotMinCount"])
print()
print(f"What the default would hide: {hidden['total']} instead of {hidden['insteadOf']} "
      f"{hidden['unit']} -- short by {hidden['shortBy']}.")
print(hidden["note"])
print()
print("gap reported:", holed["gaps"])
for name, ok in holed["checks"].items():
    print(f"  {'OK ' if ok else 'DIFF'} {name}")


## What this does not show

- **That adding up is right for these columns.** It is right for counts and wrong for rates. Deaths per 1,000 in three
  sub-districts do not add to deaths per 3,000 in one. Neither side can tell which it has been given. So Fieldwork
  offers only a sum, and does not offer averages it could not police. The protection is the **absence of the other
  option**, which no check can verify.
- **That nothing known is the right answer, rather than the honest one.** It is a refusal to guess. A practitioner
  who needs the number still has to go and find the missing cell.
- **That the three supply groups are the right way to split the data.** The both-companies group holds 300,149 of
  486,936 people. What to do with it is the practitioner's judgement.

The cell below prints these limits as the check records them.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"])
print("runtime :", report["environment"]["runtime"])
